# 09 - Live Forecasting

In [128]:
import pandas as pd
import numpy as np
import joblib
import requests

Loading Final Model

In [129]:
final_model = joblib.load(
    "../Models/final_random_forest_model.joblib"
)

feature_columns = joblib.load(
    "../Models/feature_columns.joblib"
)

print("Model loaded successfully.")
print("Number of features:", len(feature_columns))

Model loaded successfully.
Number of features: 17


Set Forecast Location

In [130]:
latitude = 7.29060
longitude = 80.63360

print("Latitude:", latitude)
print("Longitude:", longitude)

Latitude: 7.2906
Longitude: 80.6336


Fetch Live Weather Forecast

In [131]:
url = "https://api.open-meteo.com/v1/forecast"

hourly_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "cloud_cover",
    "wind_speed_10m",
    "global_tilted_irradiance",
    "diffuse_radiation",
    "sunshine_duration"
]

params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": ",".join(hourly_variables),
    "timezone": "GMT",
    "forecast_hours": 192,
    "past_hours": 1,
    "tilt": 10,
    "azimuth": 0,
    "models": "gfs_seamless"
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

response.raise_for_status()

forecast_json = response.json()

elevation = forecast_json["elevation"]

print("API request successful.")
print("Elevation:", elevation, "m")

API request successful.
Elevation: 499.0 m


 Convert Forecast Data

In [132]:
forecast_data = pd.DataFrame(
    forecast_json["hourly"]
)

forecast_data["time_utc"] = pd.to_datetime(
    forecast_data["time"],
    utc=True
)

forecast_data["time_sl"] = (
    forecast_data["time_utc"]
    .dt.tz_convert("Asia/Colombo")
)

forecast_data = forecast_data.drop(
    columns=["time"]
)

print(
    "Forecast shape:",
    forecast_data.shape
)

print(
    "First Sri Lanka time:",
    forecast_data["time_sl"].min()
)

print(
    "Last Sri Lanka time:",
    forecast_data["time_sl"].max()
)

forecast_data.head()

Forecast shape: (193, 10)
First Sri Lanka time: 2026-08-29 18:30:00+05:30
Last Sri Lanka time: 2026-09-06 18:30:00+05:30


,temperature_2m,relative_humidity_2m,precipitation,cloud_cover,wind_speed_10m,global_tilted_irradiance,diffuse_radiation,sunshine_duration,time_utc,time_sl
0,23.4,81,0.0,100,7.2,25.7,15.0,1987.38,2026-08-29 13:00:00+00:00,2026-08-29 18:30:00+05:30
1,22.4,87,0.0,100,4.8,0.0,0.0,0.00,2026-08-29 14:00:00+00:00,2026-08-29 19:30:00+05:30
2,21.7,92,0.0,100,4.1,0.0,0.0,0.00,2026-08-29 15:00:00+00:00,2026-08-29 20:30:00+05:30
3,21.2,95,0.0,100,2.3,0.0,0.0,0.00,2026-08-29 16:00:00+00:00,2026-08-29 21:30:00+05:30
4,21.0,96,0.0,100,2.6,0.0,0.0,0.00,2026-08-29 17:00:00+00:00,2026-08-29 22:30:00+05:30


In [133]:
print(
    forecast_data.columns.tolist()
)

print()

print(
    forecast_data.isna().sum()
)

['temperature_2m', 'relative_humidity_2m', 'precipitation', 'cloud_cover', 'wind_speed_10m', 'global_tilted_irradiance', 'diffuse_radiation', 'sunshine_duration', 'time_utc', 'time_sl']

temperature_2m              0
relative_humidity_2m        0
precipitation               0
cloud_cover                 0
wind_speed_10m              0
global_tilted_irradiance    0
diffuse_radiation           0
sunshine_duration           0
time_utc                    0
time_sl                     0
dtype: int64


Create Live Forecast Features

In [134]:
forecast_data["latitude"] = latitude
forecast_data["longitude"] = longitude
forecast_data["elevation"] = elevation

forecast_data["hour"] = (
    forecast_data["time_sl"].dt.hour
)

forecast_data["day_of_year"] = (
    forecast_data["time_sl"].dt.dayofyear
)

forecast_data["hour_sin"] = np.sin(
    2 * np.pi * forecast_data["hour"] / 24
)

forecast_data["hour_cos"] = np.cos(
    2 * np.pi * forecast_data["hour"] / 24
)

forecast_data["day_of_year_sin"] = np.sin(
    2 * np.pi
    * forecast_data["day_of_year"]
    / 365.25
)

forecast_data["day_of_year_cos"] = np.cos(
    2 * np.pi
    * forecast_data["day_of_year"]
    / 365.25
)

In [135]:
forecast_data["gti_lag_1"] = (
    forecast_data[
        "global_tilted_irradiance"
    ].shift(1)
)

forecast_data["cloud_cover_lag_1"] = (
    forecast_data[
        "cloud_cover"
    ].shift(1)
)

In [136]:
live_forecast = (
    forecast_data
    .iloc[1:]
    .copy()
    .reset_index(drop=True)
)

print(
    "Live forecast rows:",
    len(live_forecast)
)

print(
    "Missing GTI lag:",
    live_forecast["gti_lag_1"].isna().sum()
)

print(
    "Missing cloud lag:",
    live_forecast[
        "cloud_cover_lag_1"
    ].isna().sum()
)

Live forecast rows: 192
Missing GTI lag: 0
Missing cloud lag: 0


Verify Model Features

In [137]:
missing_features = [
    feature
    for feature in feature_columns
    if feature not in live_forecast.columns
]

print(
    "Missing model features:",
    missing_features
)

print(
    "Model input shape:",
    live_forecast[
        feature_columns
    ].shape
)

Missing model features: []
Model input shape: (192, 17)


Generate Live Solar Forecast

In [138]:
live_forecast["predicted_P"] = 0.0

active_mask = (
    live_forecast[
        "global_tilted_irradiance"
    ] > 0
)

live_forecast.loc[
    active_mask,
    "predicted_P"
] = final_model.predict(
    live_forecast.loc[
        active_mask,
        feature_columns
    ]
)

print(
    "Total forecast hours:",
    len(live_forecast)
)

print(
    "Solar-active hours:",
    active_mask.sum()
)

print(
    "Zero-GTI hours:",
    (~active_mask).sum()
)

print(
    "Negative predictions:",
    (live_forecast["predicted_P"] < 0).sum()
)

print(
    "Maximum predicted power:",
    round(
        live_forecast["predicted_P"].max(),
        2
    ),
    "W"
)

Total forecast hours: 192
Solar-active hours: 104
Zero-GTI hours: 88
Negative predictions: 0
Maximum predicted power: 664.23 W


View Hourly Solar Forecast

In [139]:
hourly_forecast = live_forecast[
    [
        "time_sl",
        "global_tilted_irradiance",
        "cloud_cover",
        "temperature_2m",
        "predicted_P"
    ]
].copy()

hourly_forecast.head(20)

,time_sl,global_tilted_irradiance,cloud_cover,temperature_2m,predicted_P
0,2026-08-29 19:30:00+05:30,0.0,100,22.4,0.000000
1,2026-08-29 20:30:00+05:30,0.0,100,21.7,0.000000
2,2026-08-29 21:30:00+05:30,0.0,100,21.2,0.000000
3,2026-08-29 22:30:00+05:30,0.0,100,21.0,0.000000
4,2026-08-29 23:30:00+05:30,0.0,100,20.8,0.000000
5,2026-08-30 00:30:00+05:30,0.0,100,20.6,0.000000
6,2026-08-30 01:30:00+05:30,0.0,100,20.4,0.000000
7,2026-08-30 02:30:00+05:30,0.0,100,20.3,0.000000
8,2026-08-30 03:30:00+05:30,0.0,100,20.2,0.000000
9,2026-08-30 04:30:00+05:30,0.0,100,20.3,0.000000


Create Daily Energy Forecast

In [140]:
live_forecast["date"] = (
    live_forecast["time_sl"].dt.date
)

hours_per_day = (
    live_forecast
    .groupby("date")
    .size()
)

complete_dates = (
    hours_per_day[
        hours_per_day == 24
    ]
    .index[:7]
)

print(
    "Complete forecast dates:",
    list(complete_dates)
)

Complete forecast dates: [datetime.date(2026, 8, 30), datetime.date(2026, 8, 31), datetime.date(2026, 9, 1), datetime.date(2026, 9, 2), datetime.date(2026, 9, 3), datetime.date(2026, 9, 4), datetime.date(2026, 9, 5)]


In [141]:
complete_forecast = live_forecast[
    live_forecast["date"].isin(
        complete_dates
    )
].copy()

print(
    "Complete forecast rows:",
    len(complete_forecast)
)

Complete forecast rows: 168


In [142]:
daily_forecast = (
    complete_forecast
    .groupby("date")
    .agg(
        predicted_energy_kwh=(
            "predicted_P",
            lambda x: x.sum() / 1000
        ),
        max_predicted_power_w=(
            "predicted_P",
            "max"
        ),
        average_cloud_cover=(
            "cloud_cover",
            "mean"
        ),
        total_precipitation_mm=(
            "precipitation",
            "sum"
        )
    )
    .reset_index()
)

daily_forecast

,date,predicted_energy_kwh,max_predicted_power_w,average_cloud_cover,total_precipitation_mm
0,2026-08-30,4.317449,652.947024,97.541667,0.0
1,2026-08-31,4.521978,657.743600,57.000000,0.0
2,2026-09-01,4.400541,645.834796,46.708333,0.0
3,2026-09-02,4.077568,586.734607,75.500000,0.0
4,2026-09-03,4.548763,659.866778,64.958333,0.0
5,2026-09-04,4.654994,664.232396,54.625000,0.0
6,2026-09-05,4.483602,639.667528,64.833333,0.0


Scale Forecast to User System

In [143]:
system_capacity_kwp = 5.0

daily_forecast["system_energy_kwh"] = (
    daily_forecast["predicted_energy_kwh"]
    * system_capacity_kwp
)

daily_forecast[
    [
        "date",
        "predicted_energy_kwh",
        "system_energy_kwh"
    ]
]

,date,predicted_energy_kwh,system_energy_kwh
0,2026-08-30,4.317449,21.587247
1,2026-08-31,4.521978,22.609888
2,2026-09-01,4.400541,22.002705
3,2026-09-02,4.077568,20.387842
4,2026-09-03,4.548763,22.743813
5,2026-09-04,4.654994,23.274972
6,2026-09-05,4.483602,22.418010


In [144]:
display_forecast = daily_forecast.copy()

display_forecast[
    "predicted_energy_kwh"
] = display_forecast[
    "predicted_energy_kwh"
].round(2)

display_forecast[
    "max_predicted_power_w"
] = display_forecast[
    "max_predicted_power_w"
].round(2)

display_forecast[
    "average_cloud_cover"
] = display_forecast[
    "average_cloud_cover"
].round(1)

display_forecast[
    "total_precipitation_mm"
] = display_forecast[
    "total_precipitation_mm"
].round(1)

display_forecast[
    "system_energy_kwh"
] = display_forecast[
    "system_energy_kwh"
].round(2)

display_forecast

,date,predicted_energy_kwh,max_predicted_power_w,average_cloud_cover,total_precipitation_mm,system_energy_kwh
0,2026-08-30,4.32,652.95,97.5,0.0,21.59
1,2026-08-31,4.52,657.74,57.0,0.0,22.61
2,2026-09-01,4.40,645.83,46.7,0.0,22.00
3,2026-09-02,4.08,586.73,75.5,0.0,20.39
4,2026-09-03,4.55,659.87,65.0,0.0,22.74
5,2026-09-04,4.65,664.23,54.6,0.0,23.27
6,2026-09-05,4.48,639.67,64.8,0.0,22.42


Forecast Summary

In [145]:
total_energy = (
    daily_forecast[
        "system_energy_kwh"
    ].sum()
)

average_daily_energy = (
    daily_forecast[
        "system_energy_kwh"
    ].mean()
)

print(
    "System capacity:",
    system_capacity_kwp,
    "kWp"
)

print(
    "7-day predicted energy:",
    round(total_energy, 2),
    "kWh"
)

print(
    "Average daily energy:",
    round(average_daily_energy, 2),
    "kWh"
)

System capacity: 5.0 kWp
7-day predicted energy: 155.02 kWh
Average daily energy: 22.15 kWh
